6) 대화 요약 메모리

In [ ]:
# [목적] ConversationSummaryMemory로 긴 대화를 요약해 보관하는 메모리 예제
# 대화가 추가될 때마다 모델이 이전 내용을 짧은 요약으로 갱신해 메모리에 저장합니다.
# 긴 기록을 모두 전달하지 않고도 핵심 문맥을 다음 대화에 유지할 때 사용합니다.
from langchain_classic.memory import ConversationSummaryMemory
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

# llm은 대화 내용을 요약할 ChatOpenAI 모델이며, return_messages=True는 결과를 메시지 목록으로 받는 설정입니다.
memory = ConversationSummaryMemory(
    llm=ChatOpenAI(model="gpt-4o", temperature=0),
    return_messages=True
)

In [ ]:
# [목적] 여행 상담 문답을 저장해 대화 요약을 점진적으로 만드는 예제
# save_context가 새 문답을 추가할 때마다 메모리는 기존 요약에 새 정보를 반영합니다.
# 여러 상담 내용을 짧은 문맥으로 압축해 이후 답변에 활용하기 위한 과정입니다.
memory.save_context(
    inputs={"human": "유럽 여행 패키지의 가격은 얼마인가요?"},
    outputs={"ai": "유럽 14박 15일 패키지의 기본 가격은 3,500유로입니다. ..."},
)

memory.save_context(
    inputs={"human": "여행 중에 방문할 주요 관광지는 어디인가요?"},
    outputs={"ai": "이 여행에서는 파리의 에펠탑, 로마의 콜로세움, 베를린의 ..."},
)

memory.save_context(
    inputs={"human": "여행자 보험은 포함되어 있나요?"},
    outputs={"ai": "네, 모든 여행자에게 기본 여행자 보험을 제공합니다. 이 보험은 ..."},
)

memory.save_context(
    inputs={"human": "항공편 좌석을 비즈니스 클래스로 업그레이드할 수 있나요? ..."},
    outputs={"ai": "항공편 좌석을 비즈니스 클래스로 업그레이드하는 것이 ..."},
)

memory.save_context(
    inputs={"human": "패키지에 포함된 호텔의 등급은 어떻게 되나요?"},
    outputs={"ai": "이 패키지에는 4성급 호텔 숙박이 포함되어 있습니다. ..."},
)

memory.save_context(
    inputs={"human": "식사 옵션에 대해 더 자세히 알려주실 수 있나요?"},
    outputs={"ai": "이 여행 패키지는 매일 아침 호텔에서 제공되는 조식을 포함하고 ..."},
)

memory.save_context(
    inputs={"human": "패키지 예약 시 예약금은 얼마인가요? 취소 정책은 어떻게 되나요?"},
    outputs={"ai": "패키지 예약 시 500유로의 예약금이 필요합니다. 취소 정책은 ..."},
)

In [ ]:
# [목적] 누적 대화가 압축된 요약 이력을 출력하는 예제
# history를 조회하면 원본 문답 전체 대신 모델이 만든 핵심 내용의 요약이 반환됩니다.
# 요약 메모리가 긴 대화를 어떻게 줄였는지 확인할 때 사용합니다.
print(memory.load_memory_variables({})["history"])

In [ ]:
# [목적] 최근 원문 대화와 이전 요약을 함께 유지하는 ConversationSummaryBufferMemory 예제
# 토큰 한도 안에서는 최근 문답을 그대로 보관하고, 넘친 오래된 내용만 요약으로 바꿉니다.
# 최신 표현은 유지하면서 긴 대화의 입력 길이를 관리할 때 사용하는 방식입니다.
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryBufferMemory

llm = ChatOpenAI()

# max_token_limit은 원문과 요약을 포함해 메모리에 유지할 최대 토큰 수를 정합니다.
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=200,
    return_messages=True,
)

In [ ]:
# [목적] 첫 상담 문답을 요약 버퍼 메모리에 저장하는 예제
# 아직 토큰 한도에 여유가 있으므로 저장한 문답은 최근 원문 대화로 유지됩니다.
# 다음 조회와 추가 저장에서 버퍼가 요약으로 전환되는 기준을 비교합니다.
memory.save_context(
    inputs={"human": "유럽 여행 패키지의 가격은 얼마인가요?"},
    outputs={
        "ai": "유럽 14박 15일 패키지의 기본 가격은 3,500유로입니다. 이 가격에는 항공료, 호텔 숙박비, 지정된 관광지 입장료가 포함되어 있습니다. 추가 비용은 선택하신 옵션 투어나 개인 경비에 따라 달라집니다."
    },
)

In [ ]:
# [목적] 첫 문답 저장 뒤 요약 버퍼 메모리의 현재 이력을 확인하는 예제
# history에는 토큰 한도 내에 있는 최근 원문 대화와 필요할 경우 이전 요약이 함께 담깁니다.
# 추가 대화를 저장하기 전 메모리의 초기 상태를 확인합니다.
memory.load_memory_variables({})["history"]

In [ ]:
# [목적] 여러 후속 문답을 추가해 요약 버퍼의 토큰 관리 동작을 확인하는 예제
# 새 문답이 쌓여 한도를 넘으면 오래된 원문은 요약되고 최근 문답은 그대로 남습니다.
# 중요한 최신 상담 내용과 전체 맥락을 함께 유지하기 위해 메모리를 확장합니다.
memory.save_context(
    inputs={"human": "여행 중에 방문할 주요 관광지는 어디인가요?"},
    outputs={
        "ai": "이 여행에서는 파리의 에펠탑, 로마의 콜로세움, 베를린의 브란덴부르크 문, 취리히의 라인폭포 등 유럽의 유명한 관광지들을 방문합니다. 각 도시의 대표적인 명소들을 포괄적으로 경험하실 수 있습니다."
    },
)

memory.save_context(
    inputs={"human": "여행자 보험은 포함되어 있나요?"},
    outputs={
        "ai": "네, 모든 여행자에게 기본 여행자 보험을 제공합니다. 이 보험은 의료비 지원, 긴급 상황 발생 시 지원 등을 포함합니다. 추가적인 보험 보장을 원하시면 상향 조정이 가능합니다."
    },
)

memory.save_context(
    inputs={
        "human": "항공편 좌석을 비즈니스 클래스로 업그레이드할 수 있나요? 비용은 어떻게 되나요?"
    },
    outputs={
        "ai": "항공편 좌석을 비즈니스 클래스로 업그레이드하는 것이 가능합니다. 업그레이드 비용은 왕복 기준으로 약 1,200유로 추가됩니다. 비즈니스 클래스에서는 더 넓은 좌석, 우수한 기내식, 그리고 추가 수하물 허용량 등의 혜택을 제공합니다."
    },
)

memory.save_context(
    inputs={"human": "패키지에 포함된 호텔의 등급은 어떻게 되나요?"},
    outputs={
        "ai": "이 패키지에는 4성급 호텔 숙박이 포함되어 있습니다. 각 호텔은 편안함과 편의성을 제공하며, 중심지에 위치해 관광지와의 접근성이 좋습니다. 모든 호텔은 우수한 서비스와 편의 시설을 갖추고 있습니다."
    },
)

In [ ]:
# [목적] 토큰 한도 적용 후 원문과 요약이 결합된 최종 이력을 확인하는 예제
# history를 조회해 오래된 내용은 요약되고 최근 내용은 원문으로 유지되었는지 살펴봅니다.
# ConversationSummaryBufferMemory의 두 가지 보관 방식을 검증하는 결과입니다.
memory.load_memory_variables({})["history"]